# WideWorldImporters City/Sales

Converted from an Azure Data Factory / Synapse **mapping data flow** (Data Flow Script) to a Microsoft Fabric PySpark notebook by the `mdf-to-fabric-spark` skill.

- **Source DFS:** `sample-mdf.txt`
- Each cell corresponds to one MDF transformation; the DataFrame variable name matches the MDF stream name.
- **DAG:** `dimcity -> cityfilter -> cityselect` and `sales -> salesfilter -> salesselect` join into `joinwithcity -> joinselect -> sinktostorage`.
- Review items are listed in the accompanying `WideWorldImportersDW_City_Sales_conversion_report.md`.

In [ ]:
# --- Parameters (from the MDF parameters{} block; overridable at run time) ---
Container       = "data"
CityFolderPath  = "WideWorldImportersDW/parquet/full/dimension_city"
SalesFolderPath = "WideWorldImportersDW/parquet/full/fact_sale"
SinkContainer   = "data"
SinkFolderPath  = "SparkTestResult"
StateFilter1    = "Texas"
StateFilter2    = "California"

# --- Storage / Fabric config (set for your environment; do NOT hardcode secrets) ---
STORAGE_ACCOUNT = "TODO_set_me"   # ADLS Gen2 account name, or use the attached lakehouse
USE_ONELAKE     = False           # True to read/write via the attached Fabric lakehouse

In [ ]:
# --- Imports (the spark session already exists in a Fabric notebook) ---
from pyspark.sql import functions as F, Window
from pyspark.sql.types import *


def storage_base(container):
    """Build the base path for a source/sink container."""
    if USE_ONELAKE:
        # Reading/writing via the attached lakehouse Files area:
        return "/lakehouse/default/Files"
    return f"abfss://{container}@{STORAGE_ACCOUNT}.dfs.core.windows.net"

## Sources

**`dimcity`** — parquet source (dimension_city)

```
source(output(CityKey as integer, WWICityID as integer, City as string,
  StateProvince as string, Country as string, Continent as string,
  SalesTerritory as string, Region as string, Subregion as string, Location as string,
  LatestRecordedPopulation as long, ValidFrom as timestamp, ValidTo as timestamp,
  LineageKey as integer),
  allowSchemaDrift: true, validateSchema: false, ignoreNoFilesFound: false,
  format: 'parquet', fileSystem: ($Container), folderPath: ($CityFolderPath)) ~> dimcity
```

In [ ]:
# allowSchemaDrift: true -> read all columns (do NOT pin the output() projection). See report.
dimcity = spark.read.format("parquet").load(f"{storage_base(Container)}/{CityFolderPath}")

**`sales`** — parquet source (fact_sale)

```
source(output(SaleKey as long, CityKey as integer, CustomerKey as integer,
  BillToCustomerKey as integer, StockItemKey as integer, InvoiceDateKey as timestamp,
  DeliveryDateKey as timestamp, SalespersonKey as integer, WWIInvoiceID as integer,
  Description as string, Package as string, Quantity as integer,
  UnitPrice as decimal(18,2), TaxRate as decimal(18,3), TotalExcludingTax as decimal(18,2),
  TaxAmount as decimal(18,2), Profit as decimal(18,2), TotalIncludingTax as decimal(18,2),
  TotalDryItems as integer, TotalChillerItems as integer, LineageKey as integer),
  allowSchemaDrift: true, validateSchema: false, ignoreNoFilesFound: false,
  format: 'parquet', fileSystem: ($Container), folderPath: ($SalesFolderPath)) ~> sales
```

In [ ]:
sales = spark.read.format("parquet").load(f"{storage_base(Container)}/{SalesFolderPath}")

## Transformations

**`salesfilter`** — filter sales rows

```
sales filter(TotalExcludingTax>700 && Profit>500) ~> salesfilter
```

In [ ]:
salesfilter = sales.filter((F.col("TotalExcludingTax") > 700) & (F.col("Profit") > 500))

**`cityfilter`** — filter cities by state

```
dimcity filter(StateProvince==$StateFilter1 || StateProvince==$StateFilter2) ~> cityfilter
```

In [ ]:
cityfilter = dimcity.filter(
    (F.col("StateProvince") == StateFilter1) | (F.col("StateProvince") == StateFilter2)
)

**`cityselect`** — project/rename city columns

```
cityfilter select(mapColumn(CityKey, City, State = StateProvince, Country, Continent,
  SalesTerritory, Region, Subregion),
  skipDuplicateMapInputs: true, skipDuplicateMapOutputs: true) ~> cityselect
```

In [ ]:
cityselect = cityfilter.select(
    F.col("CityKey"),
    F.col("City"),
    F.col("StateProvince").alias("State"),
    F.col("Country"),
    F.col("Continent"),
    F.col("SalesTerritory"),
    F.col("Region"),
    F.col("Subregion"),
)

**`salesselect`** — project/rename sales columns

```
salesfilter select(mapColumn(CityKey, InvoiceDate = InvoiceDateKey,
  DeliveryDate = DeliveryDateKey, Salesperson = SalespersonKey, Package, Quantity,
  UnitPrice, TaxRate, TotalWithoutTax = TotalExcludingTax, TaxAmount, Profit,
  TotalWithTax = TotalIncludingTax),
  skipDuplicateMapInputs: true, skipDuplicateMapOutputs: true) ~> salesselect
```

In [ ]:
salesselect = salesfilter.select(
    F.col("CityKey"),
    F.col("InvoiceDateKey").alias("InvoiceDate"),
    F.col("DeliveryDateKey").alias("DeliveryDate"),
    F.col("SalespersonKey").alias("Salesperson"),
    F.col("Package"),
    F.col("Quantity"),
    F.col("UnitPrice"),
    F.col("TaxRate"),
    F.col("TotalExcludingTax").alias("TotalWithoutTax"),
    F.col("TaxAmount"),
    F.col("Profit"),
    F.col("TotalIncludingTax").alias("TotalWithTax"),
)

**`joinwithcity`** — left join sales to city on CityKey

```
salesselect, cityselect join((salesselect@CityKey) == (cityselect@CityKey),
  joinType:'left', matchType:'exact', ignoreSpaces: false, broadcast: 'auto') ~> joinwithcity
```

Both sides expose `CityKey`; alias each side (`s`/`c`) to disambiguate. The downstream
`joinselect` keeps only the columns it names (it drops `CityKey` entirely).
`broadcast:'auto'` is left to Spark AQE (no explicit `F.broadcast`).

In [ ]:
joinwithcity = salesselect.alias("s").join(
    cityselect.alias("c"),
    F.col("s.CityKey") == F.col("c.CityKey"),
    how="left",
)

**`joinselect`** — final projection

```
joinwithcity select(mapColumn(InvoiceDate, DeliveryDate, Quantity, UnitPrice, TaxRate,
  TotalWithoutTax, TaxAmount, Profit, TotalWithTax, City, State, Country, Continent,
  SalesTerritory, Region, Subregion),
  skipDuplicateMapInputs: true, skipDuplicateMapOutputs: true) ~> joinselect
```

In [ ]:
joinselect = joinwithcity.select(
    F.col("InvoiceDate"),
    F.col("DeliveryDate"),
    F.col("Quantity"),
    F.col("UnitPrice"),
    F.col("TaxRate"),
    F.col("TotalWithoutTax"),
    F.col("TaxAmount"),
    F.col("Profit"),
    F.col("TotalWithTax"),
    F.col("City"),
    F.col("State"),
    F.col("Country"),
    F.col("Continent"),
    F.col("SalesTerritory"),
    F.col("Region"),
    F.col("Subregion"),
)

## Sinks

**`sinktostorage`** — write result parquet

```
joinselect sink(allowSchemaDrift: true, validateSchema: false, format: 'parquet',
  umask: 0022, preCommands: [], postCommands: [], skipDuplicateMapInputs: true,
  skipDuplicateMapOutputs: true, fileSystem: ($SinkContainer),
  folderPath: ($SinkFolderPath)) ~> sinktostorage
```

No write-behavior property was set in the DFS, so `overwrite` is used (matches the ADF
default of recreating the target folder). Change to `append` if the flow appended.

In [ ]:
(joinselect.write
    .format("parquet")
    .mode("overwrite")
    .save(f"{storage_base(SinkContainer)}/{SinkFolderPath}"))

## Validation (optional)

In [ ]:
# Row-count sanity checks along the DAG
print("dimcity:", dimcity.count(), "| sales:", sales.count())
print("salesfilter:", salesfilter.count(), "| cityfilter:", cityfilter.count())
print("cityselect:", cityselect.count(), "| salesselect:", salesselect.count())
print("joinselect (final):", joinselect.count())

joinselect.printSchema()

# Schema parity vs. the MDF sink projection (name, spark simpleString type).
expected = [
    ("InvoiceDate", "timestamp"),
    ("DeliveryDate", "timestamp"),
    ("Quantity", "int"),
    ("UnitPrice", "decimal(18,2)"),
    ("TaxRate", "decimal(18,3)"),
    ("TotalWithoutTax", "decimal(18,2)"),
    ("TaxAmount", "decimal(18,2)"),
    ("Profit", "decimal(18,2)"),
    ("TotalWithTax", "decimal(18,2)"),
    ("City", "string"),
    ("State", "string"),
    ("Country", "string"),
    ("Continent", "string"),
    ("SalesTerritory", "string"),
    ("Region", "string"),
    ("Subregion", "string"),
]
actual = [(f.name, f.dataType.simpleString()) for f in joinselect.schema.fields]
print("actual schema:", actual)

actual_names = [a for a, _ in actual]
missing = [c for c, _ in expected if c not in actual_names]
extra = [a for a in actual_names if a not in [c for c, _ in expected]]
assert not missing, f"Missing columns: {missing}"
assert not extra, f"Unexpected columns: {extra}"
assert actual_names == [c for c, _ in expected], "Column order differs from the MDF projection"

# A left join can legitimately leave City/State null when a sale's CityKey has no match.
print("rows with null City (left-join misses):",
      joinselect.filter(F.col("City").isNull()).count())